In [29]:
from faker import Faker
import pandas as pd
import random
from datetime import datetime, timedelta

In [30]:
fake = Faker()
random.seed(42)

### Customers Dataset

In [31]:
customers = []

segments = ["Standard", "Premium", "VIP"]

for customer_id in range(1, 1001):

    customers.append({
        "customer_id": customer_id,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.email(),
        "country": fake.country(),
        "city": fake.city(),
        "signup_date": fake.date_between(start_date="-3y", end_date="today"),
        "customer_segment": random.choice(segments)
    })

dim_customers_df = pd.DataFrame(customers)

In [32]:
dim_customers_df.head()

,customer_id,first_name,last_name,email,country,city,signup_date,customer_segment
0,1,Justin,Moore,jacksonchristina@example.net,Grenada,Greenburgh,2025-05-05,VIP
1,2,Krystal,Deleon,jennifer10@example.org,Central African Republic,Patrickbury,2023-08-01,Standard
2,3,John,Cochran,tanyasmith@example.net,Tonga,New Philip,2026-03-17,Standard
3,4,Mark,Flores,tbrown@example.org,Holy See (Vatican City State),North Stephanie,2026-04-27,VIP
4,5,Kelly,Hughes,kanderson@example.org,Holy See (Vatican City State),West Johnland,2024-09-09,Premium


### Export CSV

In [33]:
dim_customers_df.to_csv(
    "../data/raw/dim_customers.csv",
    index=False
)

### Categories Dataset

In [34]:
categories = [
    "Electronics",
    "Home",
    "Sports",
    "Fashion",
    "Books",
    "Beauty",
    "Toys"
]

dim_categories_df = pd.DataFrame({
    "category_id": range(1, len(categories) + 1),
    "category_name": categories
})

dim_categories_df

,category_id,category_name
0,1,Electronics
1,2,Home
2,3,Sports
3,4,Fashion
4,5,Books
5,6,Beauty
6,7,Toys


### Export CSV

In [35]:
dim_categories_df.to_csv(
    "../data/raw/dim_categories.csv",
    index=False
)

### Brands Dataset

In [36]:
brands = [
    "Nike",
    "Apple",
    "Samsung",
    "Adidas",
    "Sony",
    "Lego",
    "Loreal",
    "Penguin"
]

dim_brands_df = pd.DataFrame({
    "brand_id": range(1, len(brands) + 1),
    "brand_name": brands
})

dim_brands_df

,brand_id,brand_name
0,1,Nike
1,2,Apple
2,3,Samsung
3,4,Adidas
4,5,Sony
5,6,Lego
6,7,Loreal
7,8,Penguin


### Export CSV

In [37]:
dim_brands_df.to_csv(
    "../data/raw/dim_brands.csv",
    index=False
)

### Products Dataset

In [38]:
products = []

for product_id in range(1, 501):

    price = round(random.uniform(10, 1000), 2)

    products.append({
        "product_id": product_id,
        "product_name": fake.word().capitalize(),
        "category_id": random.randint(1, len(categories)),
        "brand_id": random.randint(1, len(brands)),
        "price": price,
        "cost": round(price * random.uniform(0.4, 0.8), 2),
        "created_at": fake.date_time_between(
            start_date="-2y",
            end_date="now"
        )
    })

dim_products_df = pd.DataFrame(products)

### Verify

In [39]:
dim_products_df.head()

,product_id,product_name,category_id,brand_id,price,cost,created_at
0,1,Serve,4,4,562.49,399.52,2025-06-29 06:21:43
1,2,Politics,4,5,386.57,287.63,2026-02-08 22:32:15
2,3,Attention,3,1,990.13,637.50,2025-11-24 00:38:25
3,4,Arrive,7,4,742.69,482.90,2025-02-04 23:19:27
4,5,Go,1,3,272.00,134.90,2024-09-11 11:25:00


### Export CSV

In [40]:
dim_products_df.to_csv(
    "../data/raw/dim_products.csv",
    index=False
)

### Orders Dataset

In [41]:
orders = []

order_statuses = [
    "Completed",
    "Shipped",
    "Cancelled",
    "Pending"
]

payment_methods = [
    "Credit Card",
    "PayPal",
    "Bank Transfer",
    "Crypto"
]

for order_id in range(1, 10001):

    orders.append({
        "order_id": order_id,
        "customer_id": random.randint(1, 1000),
        "order_date": fake.date_time_between(
            start_date="-2y",
            end_date="now"
        ),
        "order_status": random.choices(
            order_statuses,
            weights=[70, 20, 5, 5]
        )[0],
        "payment_method": random.choice(payment_methods),
        "shipping_country": fake.country()
    })

orders_df = pd.DataFrame(orders)

### Verify

In [42]:
orders_df.head()

,order_id,customer_id,order_date,order_status,payment_method,shipping_country
0,1,837,2025-04-05 21:38:09,Completed,Credit Card,Myanmar
1,2,795,2025-06-24 17:10:42,Shipped,PayPal,Italy
2,3,831,2025-06-03 14:29:15,Shipped,Bank Transfer,Eritrea
3,4,882,2025-04-12 23:01:36,Shipped,Crypto,Heard Island and McDonald Islands
4,5,170,2025-08-17 22:57:06,Completed,PayPal,Western Sahara


### Export CSV

In [43]:
orders_df.to_csv(
    "../data/raw/orders.csv",
    index=False
)

### Order_Items Dataset (FACT TABLE)

In [44]:
order_items = []

order_item_id = 1

for order_id in range(1, 10001):

    number_of_items = random.randint(1, 5)

    selected_products = random.sample(
        range(1, 501),
        number_of_items
    )

    for product_id in selected_products:

        product_price = dim_products_df.loc[
            dim_products_df["product_id"] == product_id,
            "price"
        ].values[0]

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": order_id,
            "product_id": product_id,
            "quantity": random.randint(1, 3),
            "unit_price": product_price,
            "discount": round(random.uniform(0, 0.3), 2)
        })

        order_item_id += 1

fact_order_items_df = pd.DataFrame(order_items)

### Verify

In [45]:
fact_order_items_df.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount
0,1,1,224,2,194.95,0.08
1,2,2,3,2,990.13,0.09
2,3,2,16,3,35.61,0.10
3,4,2,352,1,818.29,0.03
4,5,2,280,3,499.95,0.20


### Export CSV

In [46]:
fact_order_items_df.to_csv(
    "../data/raw/fact_order_items.csv",
    index=False
)